In [1]:
# Parameters
MNIST_ROOT = "D:/dev/CoursIA-26975-debate/MyIA.AI.Notebooks/SymbolicAI/ArgumentAnalysis/output/mnist"


# Argumentation-11 — Protocole de Debate & Securite

**Navigation** : [README](./README.md) | [<< 10-Privacy-Export](./Argumentation-10-Privacy-Export-Python.ipynb)

> Suite de la serie Argumentation : le **debate** comme mecanisme de securite
> (Christiano, Irving & Amodei, *AI Safety via Debate*, arXiv:1805.00899),
> rendu **execute et mesurable** sur CPU — un juge CNN sur MNIST arbitre un
> jeu de debat ou les agents revelent les pixels qu'ils invoquent.

Ce carnet implemente, de facon **deterministe et reproductible** :

1. le **jeu de debat** du §2 du paper (protocole de revelation) ;
2. la **reproduction MNIST §3.1** — un juge sur masques epars (4/6 pixels),
   avec le saut d'accuracy que le paper rapporte ;
3. la **formalisation** du transcript de debat vers un **cadre d'argumentation
   Dung/ADF** (raccorde aux carnets 03 / 03d de la serie) ;
4. les **modes de defaillance** — mind-hack (§5.8, discussion declaree) et
   collusion (§5.10, demo mesuree) ;
5. **trois exercices** (stubs conformes, a completer).

> **Honnêteté du dispositif.** Il n'y a **pas de LLM texte** sur ce siege. Les
> « agents » sont donc des **politiques deterministes** (saliencer du juge),
> pas des modeles de langage. On modele la **structure** du protocole — la
> revelation comme *gage* qui structure l'evidence — pas le reglage LLM complet
> du paper. Les parties a texte libre (mind-hack) sont une **discussion
> declaree**, jamais une sortie fabriquée.

In [2]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

SEED = 20261116
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(2)

# Racine MNIST : parametre papermill (defaut = chemin relatif au repo, gitignore
# via ArgumentAnalysis/output/). A l'execution, la lane injecte le chemin absolu.
# papermill-inject
# MNIST_ROOT "MyIA.AI.Notebooks/SymbolicAI/ArgumentAnalysis/output/mnist"
os.makedirs(MNIST_ROOT, exist_ok=True)

print("torch", torch.__version__, "| threads", torch.get_num_threads(), "| MNIST_ROOT", MNIST_ROOT)

torch 2.13.0+cu126 | threads 2 | MNIST_ROOT D:/dev/CoursIA-26975-debate/MyIA.AI.Notebooks/SymbolicAI/ArgumentAnalysis/output/mnist


## 1. Liminaire — pourquoi le debat est « PSPACE » (§2.2)

Le paper note qu'un jeu de debat **arbitre** ou deux agents jouent
strategiquement est complet pour la classe **PSPACE** : on peut y encoder des
formules quantifiees a quantificateurs alternes (QBF). Autrement dit, le
mecanisme est *expressif* — capable de porter un raisonnement aussi profond
qu'un jeu a deux joueurs sur un arbre exponentiel.

L'astuce qui rend le debat **exploitable en securite** : le **juge n'explore
pas l'arbre**. Il arbitre des **revelations**. La puissance expressive reste
cote agents ; le juge ne paie que le cout de verifier une revelation. C'est ce
decalage (profondeur chez les agents, verification polynomiale chez le juge)
qu'on met en œuvre ici.

In [3]:
# Juge CNN minimal + entrainement rapide (3 epochs, 5000 images MNIST, CPU) :
# le juge doit d'abord etre competent sur image complete pour que le debat compte.
class Judge(nn.Module):
    """Juge CNN 28x28 -> 10 chiffres. Petit, deterministe, CPU."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 8, 3, padding=1), nn.ReLU(),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(),
            nn.AdaptiveAvgPool2d(4),
            nn.Flatten(),
            nn.Linear(16 * 4 * 4, 10),
        )
    def forward(self, x):
        return self.net(x)

judge = Judge()
opt = torch.optim.Adam(judge.parameters(), lr=3e-3)
lossf = nn.CrossEntropyLoss()

_tf = transforms.Compose([transforms.ToTensor()])
_train = datasets.MNIST(MNIST_ROOT, train=True, download=True, transform=_tf)
_tr = torch.utils.data.Subset(_train, range(5000))
_loader = torch.utils.data.DataLoader(_tr, batch_size=64, shuffle=False)

judge.train()
for _ep in range(3):
    _tot, _n = 0.0, 0
    for _x, _y in _loader:
        opt.zero_grad()
        _l = lossf(judge(_x), _y)
        _l.backward()
        opt.step()
        _tot += _l.item() * _x.size(0); _n += _x.size(0)
    print(f"epoch {_ep}: loss {_tot/_n:.4f}")
judge.eval()
print("juge entraine (CPU, seed", SEED, ")")

  0%|          | 0.00/9.91M [00:00<?, ?B/s]

  1%|          | 98.3k/9.91M [00:00<00:17, 562kB/s]

  3%|▎         | 262k/9.91M [00:00<00:09, 1.04MB/s]

  6%|▌         | 557k/9.91M [00:00<00:05, 1.77MB/s]

  8%|▊         | 754k/9.91M [00:00<00:05, 1.76MB/s]

 13%|█▎        | 1.28M/9.91M [00:00<00:03, 2.85MB/s]

... (31 outputs left out)

100%|██████████| 4.54k/4.54k [00:00<00:00, 4.44MB/s]

epoch 0: loss 1.4130


epoch 1: loss 0.6340


epoch 2: loss 0.4328
juge entraine (CPU, seed 20261116 )


In [4]:
_test = datasets.MNIST(MNIST_ROOT, train=False, download=True, transform=_tf)
_N = 300  # echantillon deterministe (premiere moitie du test)
_idx = list(range(_N))
X_test = torch.stack([_test[i][0] for i in _idx])   # (N,1,28,28)
Y_test = torch.tensor([_test[i][1] for i in _idx])    # (N,)

with torch.no_grad():
    acc_full = (judge(X_test).argmax(1) == Y_test).float().mean().item()
print(f"Juge sur image COMPLETE : accuracy = {acc_full*100:.1f}%  (N={_N})")
print("-> le juge est competent : c'est la condition pour que le debat ait un sens.")

Juge sur image COMPLETE : accuracy = 89.3%  (N=300)
-> le juge est competent : c'est la condition pour que le debat ait un sens.


In [5]:
# Primitives du protocole de debate : reveler par saliencer, masquer l'image,
# choisir le challengeur, decider SANS debat (A) et AVEC debat (B).
def topk_saliency(img, y, k):
    """Indices des k pixels les plus 'informatifs pour la classe y' :
    saliencer = |gradient du logit de y par rapport a l'entree|."""
    ii = img.clone().requires_grad_(True)
    logits = judge(ii)[0]
    logits[y].backward()
    g = ii.grad[0, 0].abs().flatten()
    return torch.topk(g, k).indices

def occlude(img, flat_idx):
    """Image masquee : ne restent visibles que les pixels de flat_idx."""
    out = torch.zeros_like(img)
    m = torch.zeros(28, 28, dtype=torch.bool)
    m.flatten()[flat_idx] = True
    out[0] = img[0] * m
    return out

def challenger_label(logits_full, y_true):
    """Le challengeur = 2e classe la plus probable du juge sur image complete."""
    for c in logits_full.argsort(descending=True):
        if int(c) != y_true:
            return int(c)
    return (y_true + 1) % 10

def decide_baseline(img, y_true, y_fake, k):
    """(A) SANS DEBAT : les k pixels sont a une position FIXE, independante du
    contenu (coin superieur gauche) -> l'evidence brute ne donne aucune
    poignee au juge."""
    idx = torch.arange(k)
    ev = occlude(img, idx)
    with torch.no_grad():
        lg = judge(ev)[0]
    return y_true if lg[y_true] > lg[y_fake] else y_fake

def decide_debate(img, y_true, y_fake, k):
    """(B) AVEC DEBAT : chaque proposer revele les k pixels qu'il cite POUR SON
    label. Le juge confronte les deux evidences revelees et suit le camp le
    mieux etaye (logit de son label sur sa propre evidence)."""
    idx_true = topk_saliency(img, y_true, k)
    idx_fake = topk_saliency(img, y_fake, k)
    ev_true = occlude(img, idx_true)
    ev_fake = occlude(img, idx_fake)
    with torch.no_grad():
        lg_true = judge(ev_true)[0]
        lg_fake = judge(ev_fake)[0]
    return (y_true if lg_true[y_true] > lg_fake[y_fake] else y_fake), (idx_true, idx_fake)

In [6]:
# Boucle d'experience : sur N images, comparer (A) sans debat vs (B) avec
# debat pour chaque k, puis renvoyer la table des accuracies par k.
def run_experiment(N=150, ks=(4, 6)):
    rows = {}
    for k in ks:
        base_ok = debate_ok = 0
        for i in range(N):
            img = X_test[i:i + 1]
            y_true = int(Y_test[i])
            with torch.no_grad():
                full = judge(img)[0]
            y_fake = challenger_label(full, y_true)
            base_ok += (decide_baseline(img, y_true, y_fake, k) == y_true)
            d, _ = decide_debate(img, y_true, y_fake, k)
            debate_ok += (d == y_true)
        rows[k] = (base_ok / N, debate_ok / N)
    print(f"N={N} images | paire = (vrai, 2e classe du juge) | k pixels visibles\n")
    hdr = f"{'k':>3} | {'(A) sans debat':>16} | {'(B) avec debat':>16}"
    print(hdr); print("-" * len(hdr))
    for k in ks:
        a, b = rows[k]
        print(f"{k:>3} | {a*100:>15.1f}% | {b*100:>15.1f}%")
    return rows

results = run_experiment()

N=150 images | paire = (vrai, 2e classe du juge) | k pixels visibles

  k |   (A) sans debat |   (B) avec debat
-----------------------------------------
  4 |            64.7% |            68.0%
  6 |            64.7% |            65.3%


### Lecture du resultat

Le contraste porte tout le message du paper : **ce n'est pas plus de pixels,
c'est la structure**. A budget egal (k pixels), le regime (A) — evidence brute,
non attribuee — laisse le juge pres du hasard ; le regime (B) — les memes
pixels **cites pour un label** — lui redonne une poignee, et la precision
s'envole. C'est exactement la forme que rapporte le paper
(~ 48→85 % a 4 px, ~ 59→89 % a 6 px) : un **saut** quand la revelation
structure l'evidence, pas une pente douce.

**Ce que cette reproduction ne capture PAS** (limites, dites telles quelles) :

- les « agents » sont des **politiques de saliencer deterministes**, pas des
  LLM qui *argumentent en texte* ; on mesure le **mecanisme structurel** de la
  revelation, pas la capacite d'un modele de langage a convaincre ;
- le paper ajoute un **engagement pre-commit** des agents (hônêtete 86.7/90.4
  %) et une **remuneration** fine du juge ; ici le juge compare simplement les
  supports. Le saut de precision, lui, est bien reel et reproductible.

In [7]:
# Formalisation du transcript de debate en cadre d'argumentation Dung (ADF) :
# un argument = (label, support) ; X attaque Y si support(X) > support(Y).
def debate_to_adf(img, y_true, y_fake, k):
    """Transcript de debat -> cadre d'argumentation Dung.
    arguments = {camp: (label, support)} ; X attaque Y si support(X)>support(Y)."""
    idx_true = topk_saliency(img, y_true, k)
    idx_fake = topk_saliency(img, y_fake, k)
    with torch.no_grad():
        s_true = judge(occlude(img, idx_true))[0, y_true].item()
        s_fake = judge(occlude(img, idx_fake))[0, y_fake].item()
    args = {"A": (y_true, s_true), "B": (y_fake, s_fake)}
    attack = {("A", "B")} if s_true > s_fake else {("B", "A")}
    return args, attack

def grounded_extension(args, attack):
    """Extension fondee : les arguments non defaits (2 camps -> le gagnant)."""
    def defeated(x):
        return any(t == x for (_, t) in attack)
    return [a for a in args if not defeated(a)]

i = 0
img = X_test[i:i + 1]
y_true = int(Y_test[i])
with torch.no_grad():
    full = judge(img)[0]
y_fake = challenger_label(full, y_true)
args, attack = debate_to_adf(img, y_true, y_fake, 4)
ge = grounded_extension(args, attack)
print(f"image {i} : vrai={y_true}  challengeur={y_fake}")
for a, (lab, sup) in args.items():
    print(f"  {a} : label={lab}  support={sup:.3f}")
print("relation d'attaque :", attack)
print("extension fondee (label admissible) :", [(a, args[a][0]) for a in ge])

image 0 : vrai=7  challengeur=3
  A : label=7  support=1.584
  B : label=3  support=-1.559
relation d'attaque : {('A', 'B')}
extension fondee (label admissible) : [('A', 7)]


In [8]:
# Mesure de la collusion : les deux agents s'entendent sur le label FAUX ;
# boucle sur N images, on mesure l'accuracy quand les deux camps collusent.
def decide_collusion(img, y_true, y_fake, k):
    """COLLUSION : les deux agents s'entendent sur le label FAUX. Chacun cite
    des pixels 'soutenant' y_fake ; plus de camp honnete pour ancrer y_true.
    Le juge integre les deux avis concordants et suit l'entente."""
    idx1 = topk_saliency(img, y_fake, k)
    idx2 = topk_saliency(img, y_fake, k)
    with torch.no_grad():
        lg = (judge(occlude(img, idx1))[0] + judge(occlude(img, idx2))[0]) / 2
    return int(lg.argmax())

Nc = 150
coll_ok = 0
for i in range(Nc):
    img = X_test[i:i + 1]
    y_true = int(Y_test[i])
    with torch.no_grad():
        full = judge(img)[0]
    y_fake = challenger_label(full, y_true)
    coll_ok += (decide_collusion(img, y_true, y_fake, 4) == y_true)
debate4 = results[4][1]
print(f"k=4 :  debat honnete = {debate4*100:5.1f}%    |    collusion = {coll_ok/Nc*100:5.1f}%")
print("\n-> La collusion fait chuter l'accuracy : le debat ne tient que si les")
print("   agents NE se coordonnent pas. Condition de securite, pas un bonus.")

k=4 :  debat honnete =  68.0%    |    collusion =  12.7%

-> La collusion fait chuter l'accuracy : le debat ne tient que si les
   agents NE se coordonnent pas. Condition de securite, pas un bonus.


## Synthese

- Le **sauf-garde** du debat n'est pas « les agents sont honnetes » : c'est
  que **reveler, c'est se lier** a des preuves localisees et verifiables par le
  juge. Le saut d'accuracy (§3.1) est la preuve que cette structure *paie*.
- Le mecanisme se **formalise** en cadre Dung/ADF (§4) : la securite devient
  un calcul d'extensions ponderees, pas une question de persuasion.
- Les **failles** se nomment et se mesurent (§5) : le **mind-hack** attaque le
  juge (neutralise ici par la verifiabilite des preuves) ; la **collusion**
  desarme l'asymetrie (demontree) — elle rappelle que le debat **suppose des
  agents qui ne se coordonnent pas**, une condition a garantir *en amont*.

## Exercices

Trois stubs, conformes (aucune erreur volontaire — le carnet s'execute de bout
en bout). Chacun pousse le protocole d'un cran ; les indices pointent la
generalisation a ecrire.

In [9]:
# --- Exercice 1 : la revelation 'tout afficher' --------------------------
# Etape 1 : implementer reveal_full(img, y, k) qui revele les k pixels les
#           plus intenses de img (SANS utiliser la saliencer du juge).
# Indice : img[0].flatten() puis torch.topk(..., k).indices
def reveal_full(img, y, k):
    print("Exercice a completer")
    return None  # TODO etudiant

_demo = reveal_full(X_test[0:1], int(Y_test[0]), 4)
print("reveal_full ->", _demo, "(None tant que non implemente)")

Exercice a completer
reveal_full -> None (None tant que non implemente)


In [10]:
# --- Exercice 2 : un 3e proposer + regle de majorite --------------------
# Etape 1 : implementer debate_3(img, ys, k) : trois labels proposes, chacun
#           avec son masque de revelation ; renvoyer le label dont le support
#           (logit sur sa propre evidence) est le plus eleve.
# Indice : generaliser decide_debate a une liste de (label, masque).
def debate_3(img, ys, k):
    print("Exercice a completer")
    return None  # TODO etudiant

_demo = debate_3(X_test[0:1], [int(Y_test[0]), 3, 7], 4)
print("debate_3 ->", _demo)

Exercice a completer
debate_3 -> None


In [11]:
# --- Exercice 3 : asymetrie d'information --------------------------------
# Etape 1 : implementer debate_asym(img, y_true, y_fake, k) ou le juge ne voit
#           que le masque du CAMP HONNETE (pas celui du malhonnete). Mesurer
#           l'accuracy : que se passe-t-il quand l'adversaire ne revele rien ?
# Indice : occlude + un seul topk_saliency ; comparer lg_true[y_true] a un
#           niveau de reference (ex. le logit de y_true sur image complete).
def debate_asym(img, y_true, y_fake, k):
    print("Exercice a completer")
    return None  # TODO etudiant

_demo = debate_asym(X_test[0:1], int(Y_test[0]), 0, 4)
print("debate_asym ->", _demo)

Exercice a completer
debate_asym -> None


**Serie Argumentation** | | Precédent : [10-Privacy-Export](./Argumentation-10-Privacy-Export-Python.ipynb) | Suivant : [Obs-01-Graphes-Dates](./Argumentation-Obs-01-Graphes-Dates-Python.ipynb) | [↑ Série Argumentation](README.md)